In [ ]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt 
import matplotlib as mpl
import tifffile as tf
import os
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
#import images
os.chdir(r'')#folder dir of images to unmix
channel_dict = {}
i=1
for tiffile in os.listdir():
    if tiffile.endswith('.tif'):
        print(tiffile)
        image = np.array(tf.imread(tiffile), dtype=np.float32)-27
        image = image * np.where(image > 0, 1, 0)
        #print((image.max(),image.min()))
        channel_dict[f'channel_{i}_{tiffile.split('.')[0]}'] = image
        i+=1


In [ ]:
calibration_matrix =  np.array([[0.35,0.1,0.05],
                                [0.2,0.5, 0.05],
                                [0.1, 0.3, 0.4]])

In [ ]:
def total_variation(x, H, W):
    x = x.view(-1, H, W)  # (T, H, W)
    tv_h = torch.sum((x[:, 1:, :] - x[:, :-1, :])**2)
    tv_w = torch.sum((x[:, :, 1:] - x[:, :, :-1])**2)
    return tv_h + tv_w
image_stack_np = np.stack(list(channel_dict.values()))  # shape: (n_channels, H, W)
calibration_matrix_np = calibration_matrix.astype(np.float32)  # shape: (n_channels, n_targets)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

image_stack = torch.tensor(image_stack_np, dtype=torch.float32, device=device)  # (C, H, W)
A = torch.tensor(calibration_matrix_np, dtype=torch.float32, device=device)     # (C, T)

C, H, W = image_stack.shape
T = A.shape[1]

# Flatten image for batch processing
pixels = image_stack.view(C, -1).T  # shape: (H*W, C)

# Initialize concentrations (H*W, T)
X = torch.zeros((pixels.shape[0], T), device=device, requires_grad=True)

optimizer = torch.optim.Adam([X], lr=0.1)
num_iters = 900
lambda_l1 = 0.00000   # controls cross-talk (sparsity)
lambda_tv = 0.00000  # controls smoothness

for _ in range(num_iters):
    optimizer.zero_grad()
    pred = torch.matmul(A, X.T).T  # (H*W, C)
    data_loss = torch.nn.functional.mse_loss(pred, pixels)
    l1_reg = torch.norm(X, p=1)  # (H*W, T) → scalar
    tv_reg = total_variation(X.T, H, W)
    loss = data_loss + lambda_l1 * l1_reg + lambda_tv * tv_reg
    loss.backward()
    with torch.no_grad():
        X.data = torch.relu(X.data)  # Enforce non-negativity
    optimizer.step()

# Reshape to (T, H, W)
multichannel_image = X.T.view(T, H, W).detach().cpu().numpy()

In [ ]:
#Get metadata from saved CZI file
from pylibCZIrw import czi

czi_path = r"C:\Users\dan20\OneDrive - Johann Wolfgang Goethe Universität\Module\Masterarbeit\Aufnahmen\Zwiebel\onion_zstack_789.czi"

# --- Properly use the context manager ---
with czi.open_czi(czi_path) as reader:
    x,y,z = reader.metadata['ImageDocument']['Metadata']['Scaling']['Items']['Distance']
    metadata_complete = reader.metadata['ImageDocument']['Metadata']['Experiment']['ExperimentBlocks']['AcquisitionBlock']['AcquisitionModeSetup'].keys()
    objective=reader.metadata['ImageDocument']['Metadata']['Experiment']['ExperimentBlocks']['AcquisitionBlock']['AcquisitionModeSetup']['Objective']
    PixelPeriod = reader.metadata['ImageDocument']['Metadata']['Experiment']['ExperimentBlocks']['AcquisitionBlock']['AcquisitionModeSetup']['PixelPeriod']
    zoom = reader.metadata['ImageDocument']['Metadata']['Experiment']['ExperimentBlocks']['AcquisitionBlock']['AcquisitionModeSetup']['ZoomX']
    pixelsize=float(x['Value'])
    bidirectional = reader.metadata['ImageDocument']['Metadata']['Experiment']['ExperimentBlocks']['AcquisitionBlock']['AcquisitionModeSetup']['BiDirectional']
    #Get OME-XML metadata
    print(metadata_complete)  # Print the OME-XML metadata

In [ ]:
#write tiff file with metadata
with tf.TiffWriter(r'C:\Users\dan20\OneDrive - Johann Wolfgang Goethe Universität\Module\Masterarbeit\Aufnahmen\CACO2\Cell3\unmixed_image.tif', bigtiff=True) as tif:
    metadata = {
    'axes': 'TCZYX',
    'SignificantBits': 8,
    'SizeT': 1,
    'SizeC': 3,
    'SizeZ': 1,
    'SizeY': image_stack_np.shape[1],
    'SizeX': image_stack_np.shape[2],
    'PhysicalSizeX': pixelsize*10**6,
    'PhysicalSizeXUnit': 'µm',
    'PhysicalSizeY': pixelsize*10**6,
    'PhysicalSizeYUnit': 'µm',
    'PixelPeriod': PixelPeriod,
    'RTZoom': zoom,
    'Objective': objective,
    'Channel': {'Name': ['DNA', 'Protein', 'Lipids'],},
    #'Plane': {'PositionX': [0.0] * 16, 'PositionXUnit': ['µm'] * 16},
    'Description': 'A multi-dimensional, multi-resolution image',
    'MapAnnotation': {  # for OMERO
    'Namespace': 'openmicroscopy.org/PyramidResolution',
    '1': '256 256',
    '2': '128 128', },}
    options = dict(
        resolution=(1/(pixelsize*1e2), 1/(pixelsize*1e2)),
        resolutionunit = 'CENTIMETER',
        photometric='minisblack'
    )
    tif.write(multichannel_image.astype(np.float32),
        metadata=metadata,**options)
#print(image_stack_np.reshape((1, 1, 5, 1024, 1024)).shape)